# 01 — Simulation Pipeline

This notebook executes the full 13-phase governance friction simulation.
All computation is delegated to `scripts/run_all.py`; this notebook is orchestration only.

In [1]:
import os
import sys
from pathlib import Path

# Resolve repo root: NotebookClient sets cwd to the root, but interactive runs may start in notebooks/.
def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "scripts" / "run_all.py").is_file():
        return cwd
    if (cwd.parent / "scripts" / "run_all.py").is_file():
        return cwd.parent
    raise RuntimeError(
        "Cannot find repository root (missing scripts/run_all.py). cwd=%s" % cwd
    )


_root = _repo_root()
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("Working directory:", _root)

Working directory: C:\Users\Walter.Brown\EAA_Portfolio\ethical-alpha-audit-paper-5-sensitivity-study


In [2]:
# Execute full simulation pipeline (or verify artefacts when orchestrated after reproduce_all Step 1)
import json
import os
from pathlib import Path

if os.environ.get("P5_SKIP_EMBEDDED_SIM") == "1":
    print(
        "P5_SKIP_EMBEDDED_SIM=1: skipping scripts.run_all.main() "
        "(outputs already produced by reproduce_all Step 1)."
    )
    required = [
        "outputs/raw/scenario_results.csv",
        "outputs/processed/simulation_summary.json",
        "outputs/figures/fig1_frontier.png",
    ]
    missing = [p for p in required if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Missing expected outputs: " + ", ".join(missing))
    summary = json.loads(Path("outputs/processed/simulation_summary.json").read_text(encoding="utf-8"))
    print("Loaded simulation_summary.json (keys):", list(summary.keys()))
else:
    from scripts.run_all import main

    summary = main()

P5_SKIP_EMBEDDED_SIM=1: skipping scripts.run_all.main() (outputs already produced by reproduce_all Step 1).
Loaded simulation_summary.json (keys): ['manifest', 'validation_report', 'inference_results', 'replication_report', 'compensatory_comparison', 'table1_data', 'lifecycle_summary']


In [3]:
# Display validation summary
import json
from pathlib import Path
val_report = json.loads(Path('outputs/logs/validation_report.json').read_text())
print(f"Validation: {val_report['n_passed']}/{val_report['n_tests']} tests passed")
for name, result in val_report['details'].items():
    status = 'PASS' if result['passed'] else 'FAIL'
    print(f"  {status}: {name}")

Validation: 7/7 tests passed
  PASS: safety_monotonicity
  PASS: bias_subgroup_harm
  PASS: calibration_abstention
  PASS: drift_behaviour
  PASS: gaming_behaviour
  PASS: observation_noise_reduction
  PASS: value_bounds


In [4]:
# Display replication summary
rep_report = json.loads(Path('reproducibility/replication_report.json').read_text())
print(f"Replication: {rep_report['summary']}")
for metric, vals in rep_report['metric_comparisons'].items():
    print(f"  {metric}: diff={vals['absolute_difference']}, passed={vals['passed']}")

Replication: PASS - All metrics within tolerance
  unsafe_detection_rate: diff=0.0, passed=True
  safe_throughput: diff=0.0, passed=True
  false_negative_harm: diff=0.0, passed=True
  mean_total_friction: diff=0.0, passed=True


In [5]:
# Display run manifest
manifest = json.loads(Path('reproducibility/run_manifest.json').read_text())
print(f"Plan hash: {manifest['plan_hash'][:16]}...")
print(f"Total evaluations: {manifest['total_evaluations']}")
print(f"Pareto solutions: {manifest['nsga2_pareto_solutions']}")
print(f"Validation passed: {manifest['validation_passed']}")
print(f"Replication passed: {manifest['replication_passed']}")
print(f"Elapsed: {manifest['elapsed_seconds']}s")

Plan hash: 1fba2e2435e8fbf2...
Total evaluations: 100
Pareto solutions: 60
Validation passed: True
Replication passed: True
Elapsed: 573.3s


In [6]:
# Verify all expected output files exist
from pathlib import Path
expected = [
    'outputs/raw/scenario_results.csv',
    'outputs/raw/compensatory_comparison.csv',
    'outputs/processed/grid_results.json',
    'outputs/processed/pareto_solutions.csv',
    'outputs/processed/dca_results.json',
    'outputs/processed/inference_results.json',
    'outputs/processed/simulation_summary.json',
    'outputs/tables/table1_thresholds.csv',
    'outputs/tables/table2_pareto.csv',
    'outputs/figures/fig1_frontier.png',
    'outputs/figures/fig2_heatmap.png',
    'outputs/figures/fig3_sensitivity.png',
    'outputs/figures/fig4_drift.png',
    'outputs/figures/fig5_dca.png',
    'outputs/figures/fig6_scm.png',
]
for f in expected:
    assert Path(f).exists(), f'Missing: {f}'
print(f'All {len(expected)} expected output files present.')

All 15 expected output files present.
